In [ ]:
"""
═══════════════════════════════════════════════════════════════════════════════
Version 7 — Hard QAM Input , QPSK weight VGG Network  |  Best Accuracy: 75.63%
QPSK-Constrained Weights & Activations for RF / Wireless Inference
═══════════════════════════════════════════════════════════════════════════════

This version implements a VGG-style CNN on CIFAR-10 using fully QPSK-
constrained convolution and linear layers with RF-style magnitude recovery
sqrt(real² + imag²). Unlike v5/v6 soft-annealed inputs, Version 7 introduces
a hard argmin-based QAM input quantizer with Straight-Through Estimation (STE),
allowing direct snapping to QPSK / 16-QAM / 64-QAM constellations while still
preserving gradient flow during backpropagation. The major v7 fix replaces the
broken step-rounding quantizer (which collapsed QPSK inputs to zero) with a
correct nearest-level argmin snapper that works for all modulation orders.
The network supports configurable hard input modulation through:
INPUT_LEVELS = 4 (QPSK), 16 (16-QAM), or 64 (64-QAM), while all internal
weights and activations remain QPSK-constrained using BinaryConnect-style
latent weights and STE quantization.
═══════════════════════════════════════════════════════════════════════════════
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import math

# ═══════════════════════════════════════════════════════════════════
INPUT_LEVELS = 4   # 64 = 64-QAM,  16 = 16-QAM,  4 = QPSK
# ═══════════════════════════════════════════════════════════════════

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")


# ─────────────────────────────────────────────
# QAM CONSTELLATION TABLES
# ─────────────────────────────────────────────
def get_qam_levels(n_levels: int) -> torch.Tensor:
    n_per_axis = int(math.sqrt(n_levels))
    assert n_per_axis * n_per_axis == n_levels, \
        f"n_levels must be a perfect square (4, 16, 64). Got {n_levels}"
    raw = torch.arange(-(n_per_axis - 1), n_per_axis, 2, dtype=torch.float32)
    rms = raw.pow(2).mean().sqrt()
    return raw / rms


# ─────────────────────────────────────────────
# HARD QAM INPUT QUANTIZER  (STE)  — FIXED
# ─────────────────────────────────────────────
class HardQAMInput(nn.Module):
    def __init__(self, n_levels: int):
        # Registers the normalized constellation levels as a non-trainable
        # buffer so they move to the correct device automatically with the
        # model, and caches max_val (the outermost level magnitude) for use
        # as the Tanh bounding scale in the forward pass.
        super().__init__()
        levels  = get_qam_levels(n_levels)
        max_val = levels.abs().max().item()
        self.register_buffer('levels', levels)
        self.max_val    = max_val
        self.n_levels   = n_levels
        self.n_per_axis = int(math.sqrt(n_levels))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Applies two-stage hard quantization: first Tanh-bounds the input
        # to the outermost constellation level so no value can fall outside
        # the grid, then snaps each element to the nearest level via argmin
        # over all constellation points. The STE trick (x + (x_hard - x).detach())
        # makes the forward value equal to x_hard while routing the backward
        # gradient through x unchanged, so the float projection weights
        # upstream continue to receive useful gradients despite the
        # non-differentiable nearest-neighbor assignment.

        # Step 1: Tanh bounding — map ℝ → [-max_level, +max_level]
        x = torch.tanh(x) * self.max_val              # (B, 3, H, W)

        # Step 2: Argmin snap — correct for ALL QAM orders including QPSK
        #   x:      (B, 3, H, W, 1)
        #   levels: (1, 1, 1, 1, n_levels)
        dists  = (x.unsqueeze(-1) - self.levels.view(1, 1, 1, 1, -1)).abs()
        idx    = dists.argmin(dim=-1)                  # (B, 3, H, W)
        x_hard = self.levels[idx]                      # (B, 3, H, W)

        # STE: forward = x_hard, backward = identity (grad passes through x)
        return x + (x_hard - x).detach()

    def diagnostics(self, x_raw: torch.Tensor) -> dict:
        # Runs the forward pass on a sample batch inside a no-grad context
        # and returns a dictionary of sanity-check statistics: how many
        # distinct quantized values appeared in the output (should equal
        # n_per_axis for a healthy quantizer), the observed output range
        # and mean, and the exact floating-point level values seen. Used
        # before training starts to confirm the argmin fix is working
        # correctly for the chosen QAM order.
        with torch.no_grad():
            x_q = self.forward(x_raw)
        # Count unique values (allow small float tolerance)
        unique = x_q.unique()
        return {
            "n_unique_levels": unique.numel(),
            "expected_levels": self.n_per_axis,
            "min":  x_q.min().item(),
            "max":  x_q.max().item(),
            "mean": x_q.mean().item(),
            "levels_seen": [f"{v:.4f}" for v in unique.tolist()],
        }


# ─────────────────────────────────────────────
# QPSK WEIGHT/ACTIVATION QUANTIZER  (STE)
# ─────────────────────────────────────────────
class QPSKQuantize(torch.autograd.Function):
    # Custom autograd function that snaps each weight/activation component
    # to the nearest QPSK constellation point (±1/√2) in the forward pass,
    # while passing gradients straight through (STE) in the backward pass
    # so training is not blocked by the otherwise zero/undefined gradient
    # of the sign function.
    @staticmethod
    def forward(ctx, x):
        # Snaps every element to ±1/√2 and saves the input tensor so the
        # backward pass can apply the straight-through estimator.
        ctx.save_for_backward(x)
        return torch.sign(x) / math.sqrt(2)

    @staticmethod
    def backward(ctx, grad_output):
        # Straight-through estimator: returns the upstream gradient
        # unchanged, bypassing the zero derivative of the sign function
        # so latent weights continue to receive useful update signals.
        return grad_output.clone()

qpsk_quantize = QPSKQuantize.apply


# ─────────────────────────────────────────────
# DIAGNOSTICS HELPERS
# ─────────────────────────────────────────────
def sign_commitment(w: torch.Tensor) -> float:
    # Returns the fraction of latent weight values whose magnitude exceeds
    # 0.1, used as a proxy for how firmly each weight has "committed" to a
    # QPSK sign. A value close to 1.0 means nearly all weights are well away
    # from zero and will round stably to their intended constellation point.
    return (w.abs() > 0.1).float().mean().item()

def qpsk_snap_error(w_real: torch.Tensor, w_imag: torch.Tensor) -> float:
    # Computes the mean absolute difference between the latent (real-valued)
    # weight tensor and its hard-quantized QPSK counterpart, averaged over
    # both the real and imaginary components. Small values indicate that
    # latent weights have converged close to the discrete QPSK grid.
    q_real = torch.sign(w_real) / math.sqrt(2)
    q_imag = torch.sign(w_imag) / math.sqrt(2)
    return ((w_real - q_real).abs().mean() +
            (w_imag - q_imag).abs().mean()).item() / 2.0

def sign_penalty(w_real: torch.Tensor, w_imag: torch.Tensor) -> torch.Tensor:
    # Bimodal regularization term that penalizes latent weights near zero
    # by adding exp(-|w|) for each component. Minimizing this loss pushes
    # weights toward large positive or negative values, encouraging decisive
    # sign commitment and preventing the soft latent weights from stalling
    # near the QPSK decision boundary during training.
    return (torch.exp(-w_real.abs()) + torch.exp(-w_imag.abs())).mean()


# ─────────────────────────────────────────────
# QPSK CONV LAYER
# ─────────────────────────────────────────────
class QPSKConv2d(nn.Module):
    # A drop-in replacement for nn.Conv2d whose weight tensors are stored as
    # continuous latent values but snapped to the QPSK constellation at every
    # forward pass via the STE quantizer. Maintains separate real and imaginary
    # weight matrices, performs two convolutions, and combines their outputs as
    # a complex magnitude sqrt(r² + i²), giving a non-negative scalar activation
    # that captures both signal components while staying fully QPSK-constrained.
    def __init__(self, in_channels, out_channels, kernel_size,
                 padding=0, quantize_input=True):
        super().__init__()
        self.quantize_input = quantize_input
        self.w_real = nn.Parameter(
            torch.Tensor(out_channels, in_channels, kernel_size, kernel_size))
        self.w_imag = nn.Parameter(
            torch.Tensor(out_channels, in_channels, kernel_size, kernel_size))
        self.padding = padding
        self._init_weights()

    def _init_weights(self):
        # Initializes both the real and imaginary weight tensors with Kaiming
        # uniform values scaled down by 0.3 so the initial latent magnitudes
        # sit near the QPSK decision boundary, giving the sign-penalty
        # regularizer room to push weights toward commitment without
        # immediately saturating.
        nn.init.kaiming_uniform_(self.w_real, a=math.sqrt(5))
        nn.init.kaiming_uniform_(self.w_imag, a=math.sqrt(5))
        with torch.no_grad():
            self.w_real.data *= 0.3
            self.w_imag.data *= 0.3

    def forward(self, x):
        # Optionally quantizes the input activations to QPSK, then performs
        # two standard convolutions with QPSK-snapped real and imaginary
        # weight tensors and returns the per-element complex magnitude of
        # the result.
        if self.quantize_input:
            x = qpsk_quantize(x)
        w_q_real = qpsk_quantize(self.w_real)
        w_q_imag = qpsk_quantize(self.w_imag)
        out_real = F.conv2d(x, w_q_real, padding=self.padding)
        out_imag = F.conv2d(x, w_q_imag, padding=self.padding)
        return torch.sqrt(out_real ** 2 + out_imag ** 2 + 1e-8)

    def diagnostics(self):
        # Returns a dictionary of scalar health metrics for this layer:
        # what fraction of real/imaginary latent weights have committed to
        # a sign and how far the latent weights sit from the nearest QPSK
        # point on average.
        return {
            "sign_commit_real": sign_commitment(self.w_real),
            "sign_commit_imag": sign_commitment(self.w_imag),
            "snap_error":       qpsk_snap_error(self.w_real, self.w_imag),
        }

    def penalty(self):
        # Returns the bimodal sign-penalty scalar for this layer's weight
        # tensors, to be scaled by λ_conv and summed into the total
        # training loss.
        return sign_penalty(self.w_real, self.w_imag)


# ─────────────────────────────────────────────
# QPSK LINEAR LAYER
# ─────────────────────────────────────────────
class QPSKLinear(nn.Module):
    # Fully-connected analogue of QPSKConv2d: stores two separate latent
    # weight matrices (real and imaginary), quantizes both and the incoming
    # activations to QPSK at every forward pass, then returns the complex
    # magnitude of the two linear projections. Replaces standard float FC
    # layers so that the entire network — not just the convolutional backbone
    # — operates under QPSK constraints.
    def __init__(self, in_features, out_features):
        super().__init__()
        self.w_real = nn.Parameter(torch.Tensor(out_features, in_features))
        self.w_imag = nn.Parameter(torch.Tensor(out_features, in_features))
        self._init_weights()

    def _init_weights(self):
        # Mirrors the QPSKConv2d initialization strategy: Kaiming uniform
        # scaled by 0.3 to place initial latent weights near the QPSK
        # decision boundary, making the sign-penalty regularizer
        # immediately effective.
        nn.init.kaiming_uniform_(self.w_real, a=math.sqrt(5))
        nn.init.kaiming_uniform_(self.w_imag, a=math.sqrt(5))
        with torch.no_grad():
            self.w_real.data *= 0.3
            self.w_imag.data *= 0.3

    def forward(self, x):
        # Quantizes input activations to QPSK, applies two separate linear
        # projections with QPSK-snapped real and imaginary weight matrices,
        # and returns the element-wise complex magnitude of the results.
        x = qpsk_quantize(x)
        w_q_real = qpsk_quantize(self.w_real)
        w_q_imag = qpsk_quantize(self.w_imag)
        out_real = F.linear(x, w_q_real)
        out_imag = F.linear(x, w_q_imag)
        return torch.sqrt(out_real ** 2 + out_imag ** 2 + 1e-8)

    def diagnostics(self):
        # Returns the same set of scalar health metrics as
        # QPSKConv2d.diagnostics() so that all QPSK layers — convolutional
        # and fully-connected alike — can be monitored with a uniform
        # interface during the diagnostic logging pass.
        return {
            "sign_commit_real": sign_commitment(self.w_real),
            "sign_commit_imag": sign_commitment(self.w_imag),
            "snap_error":       qpsk_snap_error(self.w_real, self.w_imag),
        }

    def penalty(self):
        # Returns the bimodal sign-penalty scalar for this layer's weight
        # tensors, to be scaled by λ_fc and summed into the total
        # training loss.
        return sign_penalty(self.w_real, self.w_imag)


# ─────────────────────────────────────────────
# NETWORK
# ─────────────────────────────────────────────
class QPSKNet(nn.Module):
    # The full model: a VGG-style six-conv-layer backbone preceded by the
    # hard QAM input quantizer and followed by two QPSK fully-connected
    # layers and a single float linear head for class logits. Unlike v5/v6,
    # the input module here performs hard (not soft/annealed) quantization
    # via the corrected argmin-based snapper, and the input remains 3-channel
    # RGB rather than the 2-channel complex projection used in earlier versions.
    # Only the final classification head remains in full float precision,
    # matching standard practice in binary/quantized network research.
    def __init__(self, num_classes=10, input_levels=64):
        super().__init__()
        self.input_quantizer = HardQAMInput(input_levels)

        self.conv1 = QPSKConv2d(3,   128, 3, padding=1, quantize_input=False)
        self.bn1   = nn.BatchNorm2d(128, momentum=0.1, eps=1e-4)
        self.conv2 = QPSKConv2d(128, 128, 3, padding=1, quantize_input=True)
        self.bn2   = nn.BatchNorm2d(128, momentum=0.1, eps=1e-4)
        self.pool1 = nn.MaxPool2d(2, 2)

        self.conv3 = QPSKConv2d(128, 256, 3, padding=1, quantize_input=True)
        self.bn3   = nn.BatchNorm2d(256, momentum=0.1, eps=1e-4)
        self.conv4 = QPSKConv2d(256, 256, 3, padding=1, quantize_input=True)
        self.bn4   = nn.BatchNorm2d(256, momentum=0.1, eps=1e-4)
        self.pool2 = nn.MaxPool2d(2, 2)

        self.conv5 = QPSKConv2d(256, 512, 3, padding=1, quantize_input=True)
        self.bn5   = nn.BatchNorm2d(512, momentum=0.1, eps=1e-4)
        self.conv6 = QPSKConv2d(512, 512, 3, padding=1, quantize_input=True)
        self.bn6   = nn.BatchNorm2d(512, momentum=0.1, eps=1e-4)
        self.pool3 = nn.MaxPool2d(2, 2)

        self.fc1    = QPSKLinear(8192, 1024)
        self.bn_fc1 = nn.BatchNorm1d(1024, momentum=0.1, eps=1e-4)
        self.fc2    = QPSKLinear(1024, 1024)
        self.bn_fc2 = nn.BatchNorm1d(1024, momentum=0.1, eps=1e-4)
        self.fc_out = nn.Linear(1024, num_classes)

    def forward(self, x):
        # Defines the complete data flow from a raw RGB batch to class logits:
        # hard QAM input quantization → three QPSK conv blocks with batch
        # norm and max-pooling → flatten → two QPSK FC layers with batch
        # norm → float linear head. Unlike v5/v6 there is no β parameter
        # because the input quantizer is hard (argmin-based) rather than
        # soft/annealed, simplifying the call signature to a single tensor.
        x = self.input_quantizer(x)

        x = self.bn1(self.conv1(x))
        x = self.bn2(self.conv2(x))
        x = self.pool1(x)

        x = self.bn3(self.conv3(x))
        x = self.bn4(self.conv4(x))
        x = self.pool2(x)

        x = self.bn5(self.conv5(x))
        x = self.bn6(self.conv6(x))
        x = self.pool3(x)

        x = x.view(x.size(0), -1)
        x = self.bn_fc1(self.fc1(x))
        x = self.bn_fc2(self.fc2(x))
        return self.fc_out(x)

    def qpsk_layers(self):
        # Yields (name, layer) pairs for every QPSK-constrained layer in the
        # network (all six conv layers and both FC layers), providing a single
        # iteration point for diagnostics, penalty accumulation, and
        # BinaryConnect weight clipping without having to enumerate layers
        # manually elsewhere.
        return [
            ("conv1", self.conv1), ("conv2", self.conv2),
            ("conv3", self.conv3), ("conv4", self.conv4),
            ("conv5", self.conv5), ("conv6", self.conv6),
            ("fc1",   self.fc1),   ("fc2",   self.fc2),
        ]

    def get_all_diagnostics(self):
        # Collects and returns the diagnostics dictionary from every QPSK
        # layer in one call, keyed by layer name. Used during the periodic
        # diagnostic logging pass to print sign-commitment and snap-error
        # statistics for the entire network in a single loop without touching
        # individual layers directly.
        return {name: layer.diagnostics() for name, layer in self.qpsk_layers()}

    def total_sign_penalty(self, lambda_conv, lambda_fc):
        # Computes the network-wide bimodal sign-penalty regularization term
        # by summing each layer's penalty weighted by its corresponding λ
        # coefficient (λ_conv for convolutional layers, λ_fc for
        # fully-connected layers). The resulting scalar is added to the
        # cross-entropy loss at every step.
        penalty = torch.tensor(0.0, device=next(self.parameters()).device)
        for name, layer in self.qpsk_layers():
            lam = lambda_fc if name.startswith("fc") else lambda_conv
            penalty = penalty + lam * layer.penalty()
        return penalty


# ─────────────────────────────────────────────
# DATA LOADING
# ─────────────────────────────────────────────
def get_cifar10_loaders(batch_size=128, num_workers=2):
    # Constructs PyTorch DataLoaders for CIFAR-10 with standard augmentation
    # (random crop with padding and horizontal flip) applied to training data
    # only. Normalization uses the dataset's per-channel mean and standard
    # deviation so pixel values are zero-centered before entering the network.
    # num_workers is forced to 0 on MPS devices to avoid a known multiprocessing
    # incompatibility, and pin_memory is enabled only for CUDA where it provides
    # a measurable transfer speedup.
    mean = (0.4914, 0.4822, 0.4465)
    std  = (0.2023, 0.1994, 0.2010)
    train_transform = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    test_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    safe_workers = 0 if device.type == "mps" else num_workers
    train_set = torchvision.datasets.CIFAR10(
        root='./data', train=True,  download=True, transform=train_transform)
    test_set  = torchvision.datasets.CIFAR10(
        root='./data', train=False, download=True, transform=test_transform)
    train_loader = torch.utils.data.DataLoader(
        train_set, batch_size=batch_size, shuffle=True,
        num_workers=safe_workers, pin_memory=(device.type == "cuda"))
    test_loader  = torch.utils.data.DataLoader(
        test_set,  batch_size=batch_size, shuffle=False,
        num_workers=safe_workers, pin_memory=(device.type == "cuda"))
    return train_loader, test_loader


# ─────────────────────────────────────────────
# EVALUATION
# ─────────────────────────────────────────────
def evaluate(model, loader):
    # Runs inference over an entire DataLoader in evaluation mode (BatchNorm
    # uses running statistics, no dropout) with gradients disabled for speed.
    # Returns top-1 classification accuracy as a percentage. Unlike v5/v6,
    # no β argument is needed here because the input quantizer is hard and
    # stateless with respect to any annealing schedule.
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total   += labels.size(0)
    return 100.0 * correct / total


# ─────────────────────────────────────────────
# TRAINING
# ─────────────────────────────────────────────
def train_one_epoch(model, loader, optimizer, lambda_conv, lambda_fc,
                    log_grads=False):
    # Executes one full pass over the training DataLoader: computes the
    # combined cross-entropy + sign-penalty loss, back-propagates, optionally
    # logs per-layer gradient norms on the first batch of epoch 1 to detect
    # dead weights (including a "still broken!" warning if any layer shows no
    # gradient, which was the symptom of the step-rounding bug this version
    # fixes), applies BinaryConnect-style latent weight clipping to [-1, 1]
    # after each optimizer step, and returns the epoch-averaged CE loss,
    # penalty loss, and training accuracy.
    model.train()
    running_ce  = 0.0
    running_pen = 0.0
    correct = total = 0

    for batch_idx, (images, labels) in enumerate(loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs  = model(images)
        ce_loss  = F.cross_entropy(outputs, labels)
        pen_loss = model.total_sign_penalty(lambda_conv, lambda_fc)
        loss     = ce_loss + pen_loss
        loss.backward()

        if batch_idx == 0 and log_grads:
            print("\n  [GRADIENT CHECK — Batch 0]")
            for lname, layer in [("conv1", model.conv1), ("conv5", model.conv5),
                                  ("fc1",   model.fc1),   ("fc2",   model.fc2)]:
                gr = layer.w_real.grad
                gi = layer.w_imag.grad
                if gr is not None:
                    print(f"  {lname}: grad_real={gr.abs().mean().item():.6f}  "
                          f"grad_imag={gi.abs().mean().item():.6f}")
                else:
                    print(f"  {lname}: NO GRADIENT ← still broken!")
            print()

        optimizer.step()

        with torch.no_grad():
            for _, layer in model.qpsk_layers():
                layer.w_real.clamp_(-1.0, 1.0)
                layer.w_imag.clamp_(-1.0, 1.0)

        running_ce  += ce_loss.item()
        running_pen += pen_loss.item()
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total   += labels.size(0)

    n = len(loader)
    return running_ce / n, running_pen / n, 100.0 * correct / total


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────
def main():
    # Entry point that wires together all components: declares hyperparameters,
    # instantiates the model with the chosen INPUT_LEVELS constellation,
    # optimizer (Adam, weight_decay=0), and MultiStepLR scheduler, then runs
    # the training loop for 180 epochs. Before training begins it runs the
    # input quantizer diagnostics on a sample batch to verify the argmin fix
    # is producing the correct number of distinct output levels. At each epoch
    # it trains, evaluates, checkpoints the best model, and logs metrics.
    # Every DIAG_EVERY epochs it prints per-layer QPSK health statistics.
    # The checkpoint filename encodes the QAM order for easy identification
    # when running the same script with different INPUT_LEVELS values.
    EPOCHS        = 180
    BATCH_SIZE    = 128
    LR            = 1e-3
    LAMBDA_CONV   = 1e-4
    LAMBDA_FC     = 3e-4
    LR_MILESTONES = [100, 150]
    LR_GAMMA      = 0.1
    DIAG_EVERY    = 20

    qam_name  = {4: "QPSK", 16: "16-QAM", 64: "64-QAM"}.get(
        INPUT_LEVELS, f"{INPUT_LEVELS}-QAM")
    save_name = f"qpsk_cifar10_v7fix_{qam_name.replace('-','').lower()}_best.pth"

    print("=" * 70)
    print(f"QPSK-Quantized CNN on CIFAR-10  [v7-FIX — Hard {qam_name} Input]")
    print(f"Input        : Hard {qam_name}  "
          f"({int(math.sqrt(INPUT_LEVELS))} levels/channel)")
    print(f"Input levels : {get_qam_levels(INPUT_LEVELS).tolist()}")
    print( "Weights/Acts : QPSK-constrained  |  weight_decay: 0")
    print(f"Fix          : argmin-based snap replaces broken step-rounding")
    print("=" * 70)

    train_loader, test_loader = get_cifar10_loaders(BATCH_SIZE)
    model     = QPSKNet(num_classes=10, input_levels=INPUT_LEVELS).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=0)
    scheduler = torch.optim.lr_scheduler.MultiStepLR(
        optimizer, milestones=LR_MILESTONES, gamma=LR_GAMMA)

    total_params = sum(p.numel() for p in model.parameters())
    print(f"\nTotal parameters : {total_params:,}")

    # ── Verify input quantizer is working correctly ───────────────────
    sample_images = next(iter(test_loader))[0][:64].to(device)
    diag = model.input_quantizer.diagnostics(sample_images)
    print(f"\n[Input Quantizer Check — should show {diag['expected_levels']} unique levels]")
    print(f"  Unique levels seen : {diag['n_unique_levels']} "
          f"(expected {diag['expected_levels']}) "
          f"{'✓ OK' if diag['n_unique_levels'] == diag['expected_levels'] else '✗ BUG!'}")
    print(f"  Output range       : [{diag['min']:.4f}, {diag['max']:.4f}]")
    print(f"  Output mean        : {diag['mean']:.4f}")
    print(f"  Level values       : {diag['levels_seen']}")
    print()

    best_acc = 0.0

    for epoch in range(1, EPOCHS + 1):
        ce_loss, pen_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer,
            LAMBDA_CONV, LAMBDA_FC,
            log_grads=(epoch == 1)
        )
        test_acc = evaluate(model, test_loader)
        scheduler.step()

        if test_acc > best_acc:
            best_acc = test_acc
            torch.save(model.state_dict(), save_name)

        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch [{epoch:3d}/{EPOCHS}]  "
              f"CE: {ce_loss:.4f}  Pen: {pen_loss:.4f}  "
              f"Train: {train_acc:.2f}%  Test: {test_acc:.2f}%  "
              f"Best: {best_acc:.2f}%  LR: {current_lr:.2e}")

        if epoch % DIAG_EVERY == 0:
            print(f"\n  [DIAGNOSTICS — Epoch {epoch}]")
            for lname, d in model.get_all_diagnostics().items():
                print(f"  {lname:6s}: "
                      f"commit_r={d['sign_commit_real']:.3f}  "
                      f"commit_i={d['sign_commit_imag']:.3f}  "
                      f"snap_err={d['snap_error']:.4f}")
            print()

    print(f"\nTraining complete.  Best test accuracy: {best_acc:.2f}%")
    print(f"Model saved to: {save_name}")


if __name__ == "__main__":
    main()